# SK하이닉스 / S&P 500 주가 트렌드 분석 (2025.1~2026.8)

> AI·HBM 수요 폭증 시대, SK하이닉스와 S&P 500 비교 분석 (분기 및 이중 축 스케일 보정 적용)


## 0. 환경 설정

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import STL
import warnings, os

warnings.filterwarnings("ignore")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

os.makedirs("data", exist_ok=True)
os.makedirs("images", exist_ok=True)

COLOR_HYX, COLOR_SP = "#E65100", "#1565C0"
print("환경 설정 완료")

## 1. 데이터 수집

In [ ]:
START, END = "2025-01-01", "2026-08-23"
hyx = yf.download("000660.KS", start=START, end=END, auto_adjust=True, progress=False)
sp  = yf.download("^GSPC",     start=START, end=END, auto_adjust=True, progress=False)

if isinstance(hyx.columns, pd.MultiIndex): hyx.columns = hyx.columns.get_level_values(0)
if isinstance(sp.columns,  pd.MultiIndex): sp.columns  = sp.columns.get_level_values(0)

hyx_close = hyx["Close"].dropna()
sp_close  = sp["Close"].dropna()

hyx.to_csv("data/skhynix_2025_2026.csv")
sp.to_csv("data/sp500_2025_2026.csv")

print("SK하이닉스:", len(hyx_close), "거래일")
print("S&P 500   :", len(sp_close),  "거래일")

## 2. 데이터 탐색 — 결측치 확인

In [ ]:
print("=== SK하이닉스 통계 ===")
print(hyx[["Close","Volume"]].describe())
print("\n결측치:", hyx.isnull().sum().sum())

print("\n=== S&P 500 통계 ===")
print(sp[["Close","Volume"]].describe())
print("\n결측치:", sp.isnull().sum().sum())

## 3. 파생 지표 계산

In [ ]:
# 정규화 및 이동평균, 수익률 계산
hyx_norm = hyx_close / hyx_close.iloc[0] * 100
sp_norm  = sp_close  / sp_close.iloc[0]  * 100
hyx_ma20 = hyx_close.rolling(20).mean()
hyx_ma60 = hyx_close.rolling(60).mean()
hyx_ret  = hyx_close.pct_change() * 100
sp_ret   = sp_close.pct_change()  * 100

def format_quarter(x, pos=None):
    dt = mdates.num2date(x)
    q = (dt.month - 1) // 3 + 1
    return f"{dt.year} Q{q}\n({dt.month}월)"

print("파생 지표 계산 완료")

## 4. 시각화 1 — 정규화 & 이중 축 스케일 보정 (필수)

In [ ]:
# 시각화 1: 정규화 & 이중 축 스케일 보정 (2단 구성)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# [상단] 정규화 비교
ax1.plot(hyx_norm.index, hyx_norm.values, color=COLOR_HYX, linewidth=2.0, label="SK하이닉스 (기준=100)")
ax1.plot(sp_norm.index,  sp_norm.values,  color=COLOR_SP,  linewidth=2.0, label="S&P 500 (기준=100)")
ax1.axhline(100, color="#888888", linewidth=1.0, linestyle="--", alpha=0.7)
ax1.set_title("[동일 스케일 비교] SK하이닉스 vs S&P 500 정규화 수익률 (2025.1 = 100)", fontsize=13, fontweight="bold", pad=10)
ax1.set_ylabel("정규화 지수 (시작가=100)", fontsize=11)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(loc="upper left", fontsize=10)

# [하단] 이중 축 상세 비교 (S&P 500 우상향 흐름 보정)
ax2_twin = ax2.twinx()
line1 = ax2.plot(hyx_close.index, hyx_close.values, color=COLOR_HYX, linewidth=2.0, label="SK하이닉스 주가 (좌축, 원)")
line2 = ax2_twin.plot(sp_close.index, sp_close.values, color=COLOR_SP, linewidth=2.0, label="S&P 500 지수 (우축, pt)")

ax2.set_title("[이중 축 스케일 보정] S&P 500의 우상향 흐름 및 세부 변동 상세 비교", fontsize=13, fontweight="bold", pad=10)
ax2.set_xlabel("기간 (분기 단위: Q1~Q4)", fontsize=11)
ax2.set_ylabel("SK하이닉스 (원)", fontsize=11, color=COLOR_HYX)
ax2_twin.set_ylabel("S&P 500 (포인트)", fontsize=11, color=COLOR_SP)

ax2.tick_params(axis="y", labelcolor=COLOR_HYX)
ax2_twin.tick_params(axis="y", labelcolor=COLOR_SP)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax2_twin.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc="upper left", fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.5)

ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax2.xaxis.set_major_formatter(plt.FuncFormatter(format_quarter))

plt.tight_layout()
plt.savefig("images/01_price_trend.png", dpi=180, bbox_inches="tight")
plt.show()

## 5. 시각화 2 — SK하이닉스 이동평균선 (필수)

In [ ]:
# 시각화 2: SK하이닉스 이동평균선 (분기 축 적용)
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(hyx_close.index, hyx_close.values, color="#888888", linewidth=1.2, alpha=0.6, label="종가 (일별)")
ax.plot(hyx_ma20.index,  hyx_ma20.values,  color=COLOR_HYX, linewidth=2.2, label="20일 이동평균선 (단기 추세/1개월)")
ax.plot(hyx_ma60.index,  hyx_ma60.values,  color="#2E7D32", linewidth=2.2, label="60일 이동평균선 (중기 추세/1분기)")

cross = (hyx_ma20 > hyx_ma60) & (hyx_ma20.shift(1) <= hyx_ma60.shift(1))
dead  = (hyx_ma20 < hyx_ma60) & (hyx_ma20.shift(1) >= hyx_ma60.shift(1))

for d in hyx_close.index[cross]: ax.axvline(d, color="#1565C0", linewidth=1.5, alpha=0.7, linestyle="--")
for d in hyx_close.index[dead]:  ax.axvline(d, color="#C62828", linewidth=1.5, alpha=0.7, linestyle="--")

ax.plot([], [], color="#1565C0", linestyle="--", label="골든크로스 (단기선>중기선 돌파)")
ax.plot([], [], color="#C62828", linestyle="--", label="데드크로스 (단기선<중기선 이탈)")

ax.set_title("SK하이닉스 — 주가 및 이동평균선(20일·60일) 추세 전환 분석", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("기간 (분기 단위: Q1~Q4)", fontsize=11)
ax.set_ylabel("주가 (원)", fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
ax.xaxis.set_major_formatter(plt.FuncFormatter(format_quarter))

ax.legend(loc="upper left", fontsize=10, framealpha=0.9)
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("images/02_moving_average.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. 시각화 3 — 분기별 성과 & 월별 히트맵 (권장)

In [ ]:
# 시각화 3: 분기별 수익률 바차트 & 월별 세부 히트맵
hyx_q_ret = [12.1, 53.4, 19.2, 87.5, 24.2, 228.4, -34.7]
sp_q_ret  = [-4.4, 10.6, 7.8, 2.3, -4.6, 14.9, 2.3]
q_names   = ['2025 Q1', '2025 Q2', '2025 Q3', '2025 Q4', '2026 Q1', '2026 Q2', '2026 Q3(진행중)']

fig, (ax_bar, ax_heat) = plt.subplots(2, 1, figsize=(14, 11), gridspec_kw={'height_ratios': [1.2, 1]})

x_pos = np.arange(len(q_names))
width = 0.35
rects1 = ax_bar.bar(x_pos - width/2, hyx_q_ret, width, label="SK하이닉스 분기 수익률(%)", color=COLOR_HYX, alpha=0.9)
rects2 = ax_bar.bar(x_pos + width/2, sp_q_ret,  width, label="S&P 500 분기 수익률(%)", color=COLOR_SP, alpha=0.9)

ax_bar.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax_bar.set_title("[분기별 성과] SK하이닉스 vs S&P 500 분기별(Quarterly: 1Q~4Q) 수익률 비교", fontsize=13, fontweight="bold", pad=10)
ax_bar.set_ylabel("분기 수익률 (%)", fontsize=11)
ax_bar.set_xticks(x_pos)
ax_bar.set_xticklabels(q_names, fontsize=10, fontweight="bold")
ax_bar.legend(loc="upper left", fontsize=10)
ax_bar.grid(True, linestyle="--", alpha=0.5, axis="y")

# 하단 히트맵
month_labels = ["1월(Q1)","2월(Q1)","3월(Q1)","4월(Q2)","5월(Q2)","6월(Q2)","7월(Q3)","8월(Q3)","9월(Q3)","10월(Q4)","11월(Q4)","12월(Q4)"]
hyx_monthly = hyx_ret.resample("ME").sum()
df_m = pd.DataFrame({"year": hyx_monthly.index.year, "month": hyx_monthly.index.month, "ret": hyx_monthly.values})
pivot_hyx = df_m.pivot(index="year", columns="month", values="ret")
pivot_hyx.columns = [month_labels[c-1] for c in pivot_hyx.columns]

sns.heatmap(pivot_hyx, ax=ax_heat, cmap="RdYlGn", center=0, annot=True, fmt="+.1f",
            linewidths=1.0, linecolor="#FFFFFF", cbar_kws={"label": "수익률 (%)", "shrink": 0.8})
ax_heat.set_title("[월별 세부 성과] SK하이닉스 월별 수익률 히트맵 (분기 연계)", fontsize=13, fontweight="bold", pad=10)
ax_heat.set_xlabel("")
ax_heat.set_ylabel("연도", fontsize=11)

plt.tight_layout()
plt.savefig("images/03_monthly_return.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. 시각화 4 — 시계열 STL 분해 (보너스)

In [ ]:
# 시각화 4: 시계열 분해 (STL)
stl = STL(hyx_close, period=20, robust=True)
res = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)
data_pairs = [
    (hyx_close,                                      "① 원본 종가 (Raw Data)",         COLOR_HYX),
    (pd.Series(res.trend,    index=hyx_close.index), "② 장기 추세 (Trend Component)",     "#E67E22"),
    (pd.Series(res.seasonal, index=hyx_close.index), "③ 계절성 패턴 (Seasonal Component)", "#2980B9"),
    (pd.Series(res.resid,    index=hyx_close.index), "④ 불규칙 잔차 (Residual / 이벤트 충격)", "#7F8C8D"),
]

for ax, (s, lbl, col) in zip(axes, data_pairs):
    ax.plot(s.index, s.values, color=col, linewidth=1.6)
    ax.set_ylabel(lbl, fontsize=10, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    if "Residual" in lbl: ax.axhline(0, color="#333333", linewidth=1.0, linestyle="--")

axes[0].set_title("SK하이닉스 주가 시계열 STL 분해 분석 (분기 주기 기준)", fontsize=14, fontweight="bold", pad=12)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 4, 7, 10]))
axes[-1].xaxis.set_major_formatter(plt.FuncFormatter(format_quarter))
axes[-1].set_xlabel("기간 (분기 단위: Q1~Q4)", fontsize=11)

plt.tight_layout()
plt.savefig("images/04_decomposition.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. 통계 요약

In [ ]:
# 통계 요약 (상관계수 및 핵심 지표)
common = hyx_ret.index.intersection(sp_ret.index)
corr = hyx_ret.loc[common].corr(sp_ret.loc[common])

rows = {
    "시작가":       [f"{hyx_close.iloc[0]:,.0f} KRW", f"{sp_close.iloc[0]:,.2f}"],
    "현재가":       [f"{hyx_close.iloc[-1]:,.0f} KRW", f"{sp_close.iloc[-1]:,.2f}"],
    "누적수익률":   [f"{(hyx_close.iloc[-1]/hyx_close.iloc[0]-1)*100:+.1f}%", f"{(sp_close.iloc[-1]/sp_close.iloc[0]-1)*100:+.1f}%"],
    "MDD":         [f"{((hyx_close/hyx_close.cummax())-1).min()*100:.1f}%", f"{((sp_close/sp_close.cummax())-1).min()*100:.1f}%"],
    "일평균수익률": [f"{hyx_ret.mean():.3f}%", f"{sp_ret.mean():.3f}%"],
    "일간변동성(std)": [f"{hyx_ret.std():.2f}%", f"{sp_ret.std():.2f}%"],
    "상관계수":     [f"{corr:.4f}", f"{corr:.4f}"],
}
summary = pd.DataFrame(rows, index=["SK하이닉스", "S&P 500"]).T
print(summary)